In [1]:
import pandas as pd
import numpy as np
import altair as alt
from missingness_analyzer.type_of_missing_and_how import missing_how_type
from missingness_analyzer.missing_correlation_matrix import missing_correlation_matrix
from missingness_analyzer.suggest_imputation import suggest_imputation
alt.data_transformers.enable("vegafusion")

{'method': 'none', 'reasoning': ['Empty dataframe passed'], 'warnings': [], 'missingness_amount': 0}


DataTransformerRegistry.enable('vegafusion')

In [21]:
# Importing the data and encoding NULL values to use missingness_analyzer
raw = pd.read_csv("../../data/raw/diabetic_data.csv")

preprocessed_df = raw

preprocessed_df["admission_type_id"] = preprocessed_df["admission_type_id"].replace(6, np.nan)
preprocessed_df["discharge_disposition_id"] = preprocessed_df["discharge_disposition_id"].replace([18, 25, 16], np.nan)
preprocessed_df["admission_source_id"] = preprocessed_df["admission_source_id"].replace([15, 21, 20, 9], np.nan)

In [ ]:
# Correlation of columns with missing values
corr = missing_correlation_matrix(preprocessed_df).dropna(how = "all").dropna(axis = 1)
corr

In [22]:
# Missingness counts and type identification
missing_how_type(preprocessed_df)

This data frame have 191436 missing values, below is the number of missing values for each column:
encounter_id                    0
patient_nbr                     0
race                            0
gender                          0
age                             0
weight                          0
admission_type_id            5291
discharge_disposition_id     4691
admission_source_id           286
time_in_hospital                0
payer_code                      0
medical_specialty               0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                          0
diag_2                          0
diag_3                          0
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide      

,MCAR
target,
A1Cresult,False
acarbose,True
acetohexamide,True
admission_source_id,False
admission_type_id,False
age,True
change,True
chlorpropamide,True
citoglipton,True


In [ ]:
# Time in hospital vs. readmitted - boxplot
# Tells us that, graphically, patients spending less time in the hospital are more likely to not be readmitted. If they spend more time in the hospital, they are more likely to be readmitted (regardless of within 30 days or not)
plot1 = alt.Chart(preprocessed_df).mark_boxplot().encode(x = "readmitted", y = "time_in_hospital", color = "readmitted").properties(title = "Time Spent in Hospital Per Time of Readmission", width = 300, height = 300)
plot1

alt.Chart(...)

In [13]:
preprocessed_df["diabetesMed"].head()

0    0
1    1
2    1
3    1
4    1
Name: diabetesMed, dtype: object

In [18]:
# DiabetiesMed vs. readmitted
# This plot tells us that patients NOT on diabetes medication are much less likely to be readmitted
counts_df = (preprocessed_df.groupby(['diabetesMed', 'readmitted'])
               .size()
               .reset_index(name='count'))

alt.Chart(counts_df).mark_bar().encode(
    x=alt.X('diabetesMed:N',
            title='Diabetes Medication Prescribed',
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y('count:Q',
            title='Number of Patients'),
    color=alt.Color('readmitted:N',
                    scale=alt.Scale(
                        domain=['<30', '>30', 'NO'],
                        range=['#e63946', '#457b9d', '#2a9d8f']
                    ),
                    title='Readmission Status'),
    xOffset='readmitted:N'
).properties(
    title='Readmission Status by Diabetes Medication',
    width=350,
    height=350
)

alt.Chart(...)

In [25]:
# This plot tells us that the majority of patients in the dataset are aged 70-80, makes sense given the nature of diabetes
encounter_counts = (
    raw
    .groupby("patient_nbr")["encounter_id"]
    .count()
    .reset_index(name="encounter_count")  # this makes patient_nbr a regular column again
)

encounter_distribution = (
    encounter_counts
    .groupby("encounter_count")["patient_nbr"]
    .count()
    .reset_index(name="num_patients")
)
print(encounter_distribution)
 
encounter_by_age = (
    raw
    .groupby("age")
    .agg(
        encounter_count=("encounter_id", "count"),
        unique_patients=("patient_nbr", "nunique")
    )
    .reset_index()
)
print(encounter_by_age)

alt.Chart(encounter_by_age).transform_fold(
    ["encounter_count", "unique_patients"],
    as_=["metric", "value"]
).mark_bar().encode(
    x=alt.X("age:N", title="Age Category"),
    y=alt.Y("value:Q", title="Count"),
    color=alt.Color("metric:N", title="Metric"),
    xOffset=alt.XOffset("metric:N")
).properties(
    width=500,
    title="Encounters vs Unique Patients by Age Group"
)

    encounter_count  num_patients
0                 1         54745
1                 2         10434
2                 3          3328
3                 4          1421
4                 5           717
5                 6           346
6                 7           207
7                 8           111
8                 9            70
9                10            42
10               11            20
11               12            19
12               13            14
13               14             5
14               15             9
15               16             4
16               17             3
17               18             6
18               19             3
19               20             6
20               21             1
21               22             2
22               23             3
23               28             1
24               40             1
        age  encounter_count  unique_patients
0    [0-10)              161              154
1   [10-20)             

alt.Chart(...)

In [ ]:
# procedures vs. lab procedures 
# This tells us that for each lab procedure that was done, 0.03 procdures (i.e. surgeries) was done on the patient
procedures_vs_lab_procedures_ratio = preprocessed_df["num_procedures"].mean()/preprocessed_df["num_lab_procedures"].mean()
procedures_vs_lab_procedures_ratio

np.float64(0.031087375227188727)

In [ ]:
# Time in hospital by admission source

In [26]:
# Class imbalance
raw["readmitted"].value_counts()


readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64